# PINN-ResU2Net Image Denoising on BloodMNIST

---

## Problem

Medical microscopy images are routinely corrupted by noise during acquisition — sensor limitations, low illumination, and staining variability all introduce unwanted signal degradation. In blood cell analysis specifically, noise can obscure the morphological features (cell boundary sharpness, nucleus texture, cytoplasm contrast) that clinicians and automated systems rely on for diagnosis. Removing this noise without destroying those fine structural details is a non-trivial inverse problem: noise and signal overlap in both spatial and frequency domains.

---

## Why It Matters

Accurate blood cell classification depends on precise visual features. Even moderate additive Gaussian noise (σ = 25/255, the level used here) measurably degrades downstream classification accuracy and human interpretability. A denoiser that over-smooths (e.g. pure MSE-trained networks) trades noise for blur, erasing the very features needed for diagnosis. A physically grounded denoiser that preserves structural integrity is therefore clinically relevant.

---

## What Is Currently Done

Existing approaches fall into two broad families:

- **Classical methods** (BM3D, NLM, TV regularisation): exploit hand-crafted priors (non-local self-similarity, total variation). They are interpretable and parameter-light but struggle with complex, structured noise and tend to over-smooth fine textures.
- **Deep learning methods** (DnCNN, UNet, ResNet denoisers): learn the noise-to-clean mapping from data. They achieve strong PSNR/SSIM scores but are purely data-driven — they have no mechanism to enforce physical plausibility, and their performance degrades outside the training noise distribution. Crucially, training with MSE alone produces blurry outputs because the loss averages over pixel-level uncertainty.

Neither family directly encodes the governing physics of image diffusion into the learning process.

---

## Proposed Approach

We implement a **Physics-Informed Neural Network (PINN)** denoiser, following:
> *Osorio Quero & Crespo (2026): "Physics-Informed Neural Network for Denoising Images Using Nonlinear PDE", Electronics 15(3), 560.*

The core idea is to **embed partial differential equation (PDE) constraints directly into the training loss**, coupling data-driven learning with physics-based priors. Rather than training the network on reconstruction error alone, the loss simultaneously enforces: (1) fidelity to the clean reference, (2) consistency with the heat-equation diffusion prior, (3) physical boundary and initial conditions, and (4) structural perceptual quality.

This reduces dependence on large labeled datasets, improves generalisation, and prevents the network from producing physically implausible reconstructions.

---

## Methodology

### Dataset
**BloodMNIST** ([MedMNIST](https://medmnist.com/)) — 28×28 RGB microscopy images of blood cells across 8 morphological classes (~11.9K train / 1.7K val / 3.4K test). Gaussian noise (σ = 25/255) is added on-the-fly at each training step, so the model never sees the same corrupted sample twice.

### Architecture — ResU2Net
A lightweight encoder-decoder built from **Residual U-blocks (RSU)** — the key building block of U2Net, adapted here for 28×28 inputs:

- Each **RSU block** is itself a small nested U-Net: it encodes the input through two conv levels, reaches a dilated bottleneck for multi-scale context, decodes back, and adds the original input as a residual. This gives every stage both local and wider receptive field without stacking depth.
- The **outer encoder** (4 RSU stages + max-pool) extracts a hierarchy of features from fine texture (28×28) down to coarse structure (4×4).
- The **outer decoder** (3 RSU stages + bilinear upsample) restores spatial resolution using skip connections from the encoder — preserving the fine-grained cell morphology that MSE-only training tends to blur.
- A final **1×1 convolution + sigmoid** maps features back to a 3-channel image in [0, 1].

### Loss Function
The total training objective has six terms, directly mirroring Equation 10 from the paper:

$$L_{total} = L_{data} + \lambda_{pde}\,L_{PDE} + \lambda_{ic}\,L_{IC} + \lambda_{bc}\,L_{BC} + \lambda_{ssim}\,L_{SSIM} + \lambda_{l1}\,L_{L1}$$

| Term | Weight | Formula | Physical / Statistical Role |
|------|:------:|---------|-----------------------------|
| $L_{data}$ | 1.0 | MSE($\hat{X}$, $X_{clean}$) | Core pixel-level reconstruction fidelity |
| $L_{PDE}$ | 0.05 | $\\|\Delta\hat{X}\\|^2$ | Heat-equation prior: penalises spatial roughness inconsistent with isotropic diffusion |
| $L_{IC}$ | 0.03 | MSE($\hat{X}$, $X_{noisy}$) | Initial condition: output must not drift far from the observed (noisy) input at $t=0$ |
| $L_{BC}$ | 0.02 | MSE($\hat{X}_{border}$, $X_{noisy,border}$) | Dirichlet boundary condition: prevents hallucinated structure at image edges |
| $L_{SSIM}$ | 0.15 | $1 - \text{SSIM}(\hat{X}, X_{clean})$ | Directly optimises the structural evaluation metric; counteracts MSE-induced blur |
| $L_{L1}$ | 0.05 | L1($\hat{X}$, $X_{clean}$) | Outlier-robust sharpness; complements MSE which squares large errors |

The PDE loss uses the discrete Laplacian $\Delta$ computed via a fixed 3×3 convolution kernel. The IC and BC terms are the PINN-specific additions that distinguish this framework from a plain deep denoiser — they constrain the solution space to be physically consistent with the diffusion process.

### Evaluation Metrics
- **PSNR** (Peak Signal-to-Noise Ratio, dB) — measures pixel-level reconstruction accuracy; higher is better. Values above ~28 dB are generally considered good for this noise level.
- **SSIM** (Structural Similarity Index) — measures perceptual similarity in luminance, contrast, and structure on a [0, 1] scale; higher is better.

---

## Results and Challenges

### Results
The model achieves a **+9.42 dB PSNR gain** over the noisy input (20.67 → 30.09 dB) and raises SSIM from 0.708 to **0.949** on the 3,421-image test set, at an inference speed of **0.71 ms per image** — fast enough for real-time clinical pipelines.

### Challenges and Observations
- **PDE loss increases during training**: The heat-equation loss `||Δpred||²` measures spatial roughness of the output. Early in training the network produces near-uniform blurry predictions (low Laplacian → low PDE loss). As it learns to reconstruct sharp cell boundaries and texture, the Laplacian naturally grows. This is *expected and correct* — it reflects the productive tension between the smoothness prior (PDE term) and the sharpness objectives (SSIM + L1 terms). The small weight λ=0.05 ensures it acts as a regulariser, not a dominant constraint.
- **LR reduction spikes**: The visible spike at epoch ~28 in the validation curves corresponds to `ReduceLROnPlateau` halving the learning rate — a brief destabilisation before settling at a better optimum.
- **28×28 resolution limit**: BloodMNIST's small image size restricts how many pooling stages are feasible, limiting the depth of multi-scale feature extraction compared to the paper's 256×256 SAR images.

In [1]:
!pip install medmnist

Defaulting to user installation because normal site-packages is not writeable


In [ ]:
# Install dependencies if needed
# !pip install medmnist scikit-image torch torchvision matplotlib tqdm

import numpy as np
import matplotlib.pyplot as plt
import time

import torch

from skimage.metrics import peak_signal_noise_ratio as psnr_metric
from skimage.metrics import structural_similarity as ssim_metric

# Project modules
from dataset import NoisyDataset, get_dataloaders, NOISE_SIGMA, BATCH_SIZE
from model import ResU2Net
from losses import pinn_loss, laplacian, ssim_loss
from metrics import batch_psnr_ssim
from visualize import show_pairs, plot_history, show_test_results

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')


## Dataset and DataLoader

We wrap BloodMNIST in a custom `NoisyDataset` that:
1. Loads the clean image (normalized to [0, 1]).
2. Adds **Gaussian noise** (σ = 25/255) on the fly to create the corrupted input.

This way the model always sees fresh noise samples, acting as light data augmentation.

In [ ]:
train_loader, val_loader, test_loader = get_dataloaders(
    batch_size=BATCH_SIZE,
    sigma=NOISE_SIGMA,
    num_workers=2,
)


## Visualise DataLoader Samples

We display a batch of **clean** and corresponding **noisy** images to confirm the noise model looks reasonable.

In [ ]:
noisy_sample, clean_sample = next(iter(train_loader))
show_pairs(noisy_sample, clean_sample, n=8,
           title='BloodMNIST — Clean vs Noisy (σ=25/255)')


## Model Architecture — ResU2Net

### Building Blocks
- **ConvBNReLU**: Conv2d → BatchNorm → ReLU — standard feature extractor.
- **RSU (Residual U-block)**: A small nested U-Net inside each stage:
  - Encodes with `depth` conv layers → decodes back → adds residual from input.
  - This gives each stage multi-scale context while keeping parameter count low.

### Full ResU2Net
```
Input (3×28×28)
  ↓ Encoder: Stage1(RSU-4) → pool → Stage2(RSU-3) → pool → Stage3(RSU-2) → pool → Stage4(RSU-1)
  ↓ Decoder: Stage3d ← upsample+concat ← Stage2d ← upsample+concat ← Stage1d
  ↓ Output conv 1×1 → (3×28×28)
```
Skip connections between matching encoder-decoder stages preserve spatial detail.

In [ ]:
model = ResU2Net(in_ch=3, out_ch=3, base=16).to(DEVICE)

# Sanity check
dummy = torch.randn(2, 3, 28, 28).to(DEVICE)
with torch.no_grad():
    out = model(dummy)
assert out.shape == dummy.shape, f'Shape mismatch: {out.shape}'

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Input : {dummy.shape}  →  Output: {out.shape}')
print(f'Trainable parameters: {total_params:,}')


## Physics-Informed Loss Functions

Following the paper's composite loss (Eq. 10), we use **six terms**:

$$L_{total} = L_{data} + \lambda_{pde}\,L_{PDE} + \lambda_{ic}\,L_{IC} + \lambda_{bc}\,L_{BC} + \lambda_{ssim}\,L_{SSIM} + \lambda_{l1}\,L_{L1}$$

| Term | Weight | Formula | Role |
|------|--------|---------|------|
| $L_{data}$ | 1.0 | MSE(pred, clean) | Core pixel fidelity |
| $L_{PDE}$ | 0.05 | $\|\Delta\hat{X}\|^2$ (heat residual) | Physics prior — penalise spatial roughness |
| $L_{IC}$ | 0.03 | MSE(pred, noisy) | Initial condition — pred should not drift far from noisy input |
| $L_{BC}$ | 0.02 | MSE(pred border, noisy border) | Dirichlet BC — no hallucinated edges at image boundary |
| $L_{SSIM}$ | 0.15 | $1 - \text{SSIM}$(pred, clean) | Directly optimise the structural metric we evaluate on |
| $L_{L1}$ | 0.05 | L1(pred, clean) | Sharpness + outlier robustness |

**Why IC and BC terms help**: Without them, the network can produce low-MSE outputs that violate physical plausibility (e.g. invented edge structure). The IC term keeps the denoised image anchored to the observed input; the BC term prevents the network from hallucinating content at the image borders — both mirror exactly the PINN formulation in the paper.

**Why SSIM loss**: MSE optimisation implicitly averages uncertain regions, producing blur. The SSIM term directly penalises loss of structure and contrast, which is what PSNR/SSIM metrics measure.

In [ ]:
# Loss functions are defined in losses.py
# pinn_loss, laplacian, and ssim_loss are already imported above.
# See losses.py for full implementation and docstrings.
print('Loss functions loaded from losses.py')


## Metrics

PSNR and SSIM are computed on CPU numpy arrays in [0, 1] using `scikit-image`.
We average over the batch for a scalar result per iteration.

In [ ]:
# batch_psnr_ssim is defined in metrics.py and already imported above.
print('Metrics loaded from metrics.py')


## Training

- **Optimizer**: Adam with lr = 1e-3
- **Scheduler**: ReduceLROnPlateau — halves lr when validation loss plateaus for 5 epochs
- **Early stopping**: Stops if no improvement in validation loss for 10 epochs
- We log per-epoch: total loss, MSE, PDE, L1 sub-losses, and PSNR/SSIM — for both train and validation.

In [8]:
NUM_EPOCHS   = 500
LR           = 1e-4
PATIENCE     = 50     # early stopping patience

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5, verbose=True)

# History containers
history = {
    'train_loss': [], 'train_mse': [], 'train_pde': [],
    'train_ic':   [], 'train_bc':  [], 'train_ssim_loss': [], 'train_l1': [],
    'train_psnr': [], 'train_ssim': [],
    'val_loss':   [], 'val_mse':   [], 'val_pde':   [],
    'val_ic':     [], 'val_bc':    [], 'val_ssim_loss':   [], 'val_l1':   [],
    'val_psnr':   [], 'val_ssim':  [],
}

best_val_loss = float('inf')
epochs_no_improve = 0


def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    tot = {k: 0.0 for k in ['loss','mse','pde','ic','bc','ssim','l1']}
    psnr_list, ssim_list = [], []

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for noisy, clean in loader:
            noisy, clean = noisy.to(DEVICE), clean.to(DEVICE)
            pred = model(noisy)
            loss, comps = pinn_loss(pred, clean, noisy)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            tot['loss'] += loss.item()
            for k, v in comps.items():
                tot[k] += v

            p, s = batch_psnr_ssim(pred, clean)
            psnr_list.append(p)
            ssim_list.append(s)

    n = len(loader)
    return {k: v / n for k, v in tot.items()} | {'psnr': np.mean(psnr_list), 'ssim': np.mean(ssim_list)}


print(f'Training for up to {NUM_EPOCHS} epochs (early stop patience={PATIENCE})\n')
print(f'{"Epoch":>6} | {"Tr Loss":>8} | {"Tr PSNR":>8} | {"Tr SSIM":>8} | {"Va Loss":>8} | {"Va PSNR":>8} | {"Va SSIM":>8}')
print('-' * 70)

for epoch in range(1, NUM_EPOCHS + 1):
    tr = run_epoch(train_loader, train=True)
    va = run_epoch(val_loader,   train=False)

    for k in ['loss','mse','pde','ic','bc','l1','psnr','ssim']:
        history[f'train_{k}'].append(tr[k])
        history[f'val_{k}'].append(va[k])
    history['train_ssim_loss'].append(tr['ssim_loss'] if 'ssim_loss' in tr else tr.get('ssim', 0))
    history['val_ssim_loss'].append(va['ssim_loss']   if 'ssim_loss' in va else va.get('ssim', 0))

    scheduler.step(va['loss'])

    print(f'{epoch:>6} | {tr["loss"]:>8.4f} | {tr["psnr"]:>8.2f} | {tr["ssim"]:>8.4f} '
          f'| {va["loss"]:>8.4f} | {va["psnr"]:>8.2f} | {va["ssim"]:>8.4f}')

    # Early stopping + checkpoint
    if va['loss'] < best_val_loss:
        best_val_loss = va['loss']
        epochs_no_improve = 0
        torch.save(model.state_dict(), 'best_model.pt')
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f'\nEarly stopping at epoch {epoch}.')
            break

print('\nTraining complete. Loading best checkpoint.')
model.load_state_dict(torch.load('best_model.pt', map_location=DEVICE))

Training for up to 500 epochs (early stop patience=50)

 Epoch |  Tr Loss |  Tr PSNR |  Tr SSIM |  Va Loss |  Va PSNR |  Va SSIM
----------------------------------------------------------------------


/home/bkw6756/.local/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


     1 |   0.0697 |    16.07 |   0.7333 |   0.0368 |    19.02 |   0.8604
     2 |   0.0323 |    20.09 |   0.8739 |   0.0290 |    21.39 |   0.8807
     3 |   0.0244 |    22.43 |   0.8966 |   0.0216 |    23.58 |   0.9046
     4 |   0.0206 |    24.20 |   0.9070 |   0.0189 |    25.17 |   0.9129
     5 |   0.0189 |    25.36 |   0.9117 |   0.0176 |    26.04 |   0.9174
     6 |   0.0176 |    26.26 |   0.9170 |   0.0185 |    26.59 |   0.9139
     7 |   0.0169 |    26.82 |   0.9198 |   0.0162 |    27.13 |   0.9234
     8 |   0.0165 |    27.18 |   0.9210 |   0.0161 |    27.58 |   0.9235
     9 |   0.0161 |    27.51 |   0.9235 |   0.0156 |    27.87 |   0.9258
    10 |   0.0158 |    27.73 |   0.9249 |   0.0158 |    27.88 |   0.9251
    11 |   0.0156 |    27.88 |   0.9258 |   0.0156 |    28.05 |   0.9265
    12 |   0.0155 |    28.00 |   0.9264 |   0.0155 |    28.21 |   0.9269
    13 |   0.0153 |    28.12 |   0.9272 |   0.0156 |    28.28 |   0.9269
    14 |   0.0152 |    28.19 |   0.9279 |   0.0150 

/tmp/ipykernel_1911737/3933547966.py:82: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_model.pt', map_location=DEVICE))


<All keys matched successfully>

## Training History Plots

We visualise:
1. **Total loss** (train vs. val)
2. **Loss sub-components** (MSE, PDE, L1) for training
3. **PSNR** (train vs. val)
4. **SSIM** (train vs. val)

In [ ]:
plot_history(history)


## Test Set Evaluation

We evaluate on the held-out test set and report:
- **PSNR** and **SSIM** for **noisy input** vs. clean (baseline — what we start with)
- **PSNR** and **SSIM** for **model output** vs. clean (what we achieve)
- **Inference time** in milliseconds per image

In [10]:
model.eval()

all_psnr_noisy, all_ssim_noisy = [], []
all_psnr_pred,  all_ssim_pred  = [], []
total_images = 0

# Store a few batches for visualisation
vis_noisy, vis_clean, vis_pred = [], [], []

t_start = time.time()

with torch.no_grad():
    for i, (noisy, clean) in enumerate(test_loader):
        noisy, clean = noisy.to(DEVICE), clean.to(DEVICE)
        pred = model(noisy)

        pn, sn = batch_psnr_ssim(noisy, clean)
        pp, sp = batch_psnr_ssim(pred,  clean)
        all_psnr_noisy.append(pn);  all_ssim_noisy.append(sn)
        all_psnr_pred.append(pp);   all_ssim_pred.append(sp)
        total_images += noisy.shape[0]

        if i < 2:   # save first 2 batches for display
            vis_noisy.append(noisy.cpu())
            vis_clean.append(clean.cpu())
            vis_pred.append(pred.cpu())

t_elapsed = time.time() - t_start
ms_per_img = (t_elapsed / total_images) * 1000

print('=' * 50)
print(f'  Test set results  ({total_images} images)')
print('=' * 50)
print(f'  Noisy input  →  PSNR: {np.mean(all_psnr_noisy):.2f} dB  |  SSIM: {np.mean(all_ssim_noisy):.4f}')
print(f'  Model output →  PSNR: {np.mean(all_psnr_pred):.2f} dB  |  SSIM: {np.mean(all_ssim_pred):.4f}')
print(f'  PSNR gain    →  Δ{np.mean(all_psnr_pred) - np.mean(all_psnr_noisy):.2f} dB')
print(f'  Inference time: {ms_per_img:.3f} ms / image  (total {t_elapsed:.2f}s)')
print('=' * 50)

  Test set results  (3421 images)
  Noisy input  →  PSNR: 20.67 dB  |  SSIM: 0.7081
  Model output →  PSNR: 29.46 dB  |  SSIM: 0.9418
  PSNR gain    →  Δ8.79 dB
  Inference time: 0.716 ms / image  (total 2.45s)


## Visual Results on the Test Set

Each row shows three versions of the same image:
- **Noisy**: the corrupted input fed to the model
- **Denoised**: model output
- **Clean**: ground truth

Individual PSNR and SSIM values are shown beneath each denoised image.

In [ ]:
show_test_results(vis_noisy, vis_pred, vis_clean, n_show=10)
